# CausalCrisis v2 (Phase 3c: Heterophily-Aware Graph) - Standalone Notebook

Đây là phiên bản Notebook ĐỘC LẬP TỔNG HỢP (Standalone). 
Toàn bộ khối kiến trúc (Models, Trainers, Edge Scorer, GNN) đã được mình *nhúng trực tiếp* vào các ô lệnh bên dưới để đảm bảo Colab có thể khởi chạy lập tức mà **không cần upload folder src/** hay import lòng vòng phức tạp.

In [ ]:
# 1. Install dependencies
!pip install open_clip_torch transformers scikit-learn matplotlib seaborn pandas pyngrok

In [ ]:

# --- CONTRASTIVE LEARNING MODULES ---
class InfoNCELoss(nn.Module):
    def __init__(self, temperature=0.1):
        super().__init__()
        self.temp = temperature
        
    def forward(self, z_i, z_j):
        batch_size = z_i.size(0)
        sim_matrix = torch.matmul(z_i, z_j.T) / self.temp
        labels = torch.arange(batch_size, device=z_i.device)
        return (F.cross_entropy(sim_matrix, labels) + F.cross_entropy(sim_matrix.T, labels)) / 2.0

class GraphContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.1):
        super().__init__()
        self.info_nce = InfoNCELoss(temperature)
        
    def forward(self, z1, z2):
        z1 = F.normalize(z1, dim=-1)
        z2 = F.normalize(z2, dim=-1)
        return self.info_nce(z1, z2)

def drop_edge_randomly(adj, p=0.1):
    if p <= 0.0: return adj
    mask = torch.rand_like(adj) > p
    adj_dropped = adj * mask
    row_sum = adj_dropped.sum(dim=1, keepdim=True).clamp(min=1e-8)
    return adj_dropped / row_sum

def drop_node_feature(x, p=0.15):
    if p <= 0.0: return x
    return F.dropout(x, p=p, training=True)
# ------------------------------------

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, roc_auc_score
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import seaborn as sns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Core Methodologies & Losses

In [ ]:
# --- Gradient Reversal Layer ---
class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.clone()

    @staticmethod
    def backward(ctx, grads):
        lambda_ = ctx.lambda_
        lambda_ = grads.new_tensor(lambda_)
        dx = -lambda_ * grads
        return dx, None

def get_grl_lambda(epoch, max_epoch, warmup=5, max_lambda=1.0):
    if epoch < warmup:
        return 0.0
    progress = (epoch - warmup) / (max_epoch - warmup)
    return max_lambda * (2. / (1. + np.exp(-10. * progress)) - 1)


## 3. CausalCrisis v2 Architectures (Phase 3c Heterophily GNN)

In [ ]:

class CrossModalAdaptiveFusion(nn.Module):
    def __init__(self, hidden_dim, num_heads=4):
        super().__init__()
        self.attn_i2t = nn.MultiheadAttention(hidden_dim, num_heads, batch_first=True)
        self.attn_t2i = nn.MultiheadAttention(hidden_dim, num_heads, batch_first=True)
        self.weight_gate = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 2),
            nn.Softmax(dim=-1)
        )
        
    def forward(self, img_feat, txt_feat):
        img_seq = img_feat.unsqueeze(1)
        txt_seq = txt_feat.unsqueeze(1)
        img_t, _ = self.attn_i2t(img_seq, txt_seq, txt_seq)
        txt_i, _ = self.attn_t2i(txt_seq, img_seq, img_seq)
        img_t = img_t.squeeze(1)
        txt_i = txt_i.squeeze(1)
        fused_cat = torch.cat([img_t, txt_i], dim=-1)
        weights = self.weight_gate(fused_cat)
        w_img = weights[:, 0].unsqueeze(1)
        w_txt = weights[:, 1].unsqueeze(1)
        z_unified = torch.cat([w_img * img_t, w_txt * txt_i], dim=-1)
        return z_unified, weights

class GraphSAGELayer(nn.Module):
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        self.proj = nn.Linear(in_dim * 2, out_dim)
        
    def forward(self, x: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        h_N = torch.matmul(adj, x)
        h_cat = torch.cat([x, h_N], dim=-1)
        return F.relu(self.proj(h_cat))

class CausalGNNModule(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, dropout=0.3):
        super().__init__()
        self.conv1 = GraphSAGELayer(in_dim, hidden_dim)
        self.conv2 = GraphSAGELayer(hidden_dim, in_dim)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(in_dim)
        
    def forward(self, x: torch.Tensor, adj: torch.Tensor):
        h = self.conv1(x, adj)
        h = self.dropout(h)
        h = self.conv2(h, adj)
        return self.norm(x + h)

class ModalityProjector(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, dropout=0.3):
        super().__init__()
        self.general_proj = nn.Sequential(nn.Linear(in_dim, in_dim // 2), nn.LayerNorm(in_dim // 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(in_dim // 2, out_dim))
        self.specific_proj = nn.Sequential(nn.Linear(in_dim, in_dim // 2), nn.LayerNorm(in_dim // 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(in_dim // 2, out_dim))

    def forward(self, x: torch.Tensor):
        return self.general_proj(x), self.specific_proj(x)

class CausalDisentanglerV2(nn.Module):
    def __init__(self, in_dim: int, causal_dim: int, spurious_dim: int, dropout=0.3):
        super().__init__()
        self.causal_enc = nn.Sequential(nn.Linear(in_dim, in_dim), nn.LayerNorm(in_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(in_dim, causal_dim))
        self.spurious_enc = nn.Sequential(nn.Linear(in_dim, in_dim), nn.LayerNorm(in_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(in_dim, spurious_dim))

    def forward(self, z_unified: torch.Tensor):
        return self.causal_enc(z_unified), self.spurious_enc(z_unified)

class DomainClassifier(nn.Module):
    def __init__(self, in_dim: int, num_domains: int):
        super().__init__()
        self.net = nn.Sequential(nn.utils.spectral_norm(nn.Linear(in_dim, in_dim // 2)), nn.ReLU(), nn.utils.spectral_norm(nn.Linear(in_dim // 2, num_domains)))
    def forward(self, x: torch.Tensor):
        return self.net(x)

class EdgeHeterophilyScorer(nn.Module):
    """
    Phase 3c: Phạt cạnh GNN dị biệt (Cắt Spurious Edges)
    """
    def __init__(self, in_dim: int, num_classes: int, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim + num_classes, in_dim // 2), nn.LayerNorm(in_dim // 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(in_dim // 2, 1))
        
    def forward(self, x_i, x_j, p_i, p_j):
        feat_diff = torch.abs(x_i - x_j)
        prob_sim = p_i * p_j
        concat_feat = torch.cat([feat_diff, prob_sim], dim=-1)
        return torch.sigmoid(self.net(concat_feat)).squeeze(-1)

class CausalCrisisV2Model(nn.Module):
    def __init__(self, img_dim=1024, txt_dim=768, hidden_dim=256, causal_dim=256, spurious_dim=256, num_domains=7, num_classes=2, dropout=0.3):
        super().__init__()
        self.img_proj = ModalityProjector(img_dim, hidden_dim, dropout)
        self.txt_proj = ModalityProjector(txt_dim, hidden_dim, dropout)
        self.fusion_layer = CrossModalAdaptiveFusion(hidden_dim, num_heads=4)
        self.causal_disentangle = CausalDisentanglerV2(hidden_dim * 2, causal_dim, spurious_dim, dropout)
        self.gnn = CausalGNNModule(causal_dim, causal_dim, dropout)
        self.heterophily_scorer = EdgeHeterophilyScorer(causal_dim, num_classes, dropout)
        self.domain_classifier = DomainClassifier(causal_dim, num_domains)
        self.classifier = nn.Sequential(nn.Linear(causal_dim, causal_dim // 2), nn.ReLU(), nn.Dropout(dropout), nn.Linear(causal_dim // 2, num_classes))

    def forward(self, img_feat, txt_feat, adj=None, backdoor_xs=None, return_contrastive_loss=False):
        outputs = {}
        z_img_g, z_img_s = self.img_proj(img_feat)
        z_txt_g, z_txt_s = self.txt_proj(txt_feat)
        outputs["z_img_g"], outputs["z_img_s"] = z_img_g, z_img_s
        outputs["z_txt_g"], outputs["z_txt_s"] = z_txt_g, z_txt_s
        
        z_unified, weights = self.fusion_layer(z_img_g, z_txt_g)
        outputs['fusion_weights'] = weights
        outputs["z_unified"] = z_unified
        
        xc, xs = self.causal_disentangle(z_unified)
        outputs["xc"], outputs["xs"] = xc, xs
        outputs["domain_logits"] = self.domain_classifier(xc)
        
        if adj is not None:
            xc_graph = self.gnn(xc, adj)
            outputs["xc_graph"] = xc_graph
        else:
            outputs["xc_graph"] = xc
        
        if backdoor_xs is not None:
            M = backdoor_xs.shape[1]
            xc_expand = outputs["xc_graph"].unsqueeze(1).expand(-1, M, -1)
            combined = xc_expand + backdoor_xs 
            logits_M = self.classifier(combined)
            probs_M = torch.softmax(logits_M, dim=-1)
            expected_probs = probs_M.mean(dim=1)
            outputs["logits_ba"] = torch.log(expected_probs + 1e-8)
        else:
            xs_detached = xs.detach()
            combined = outputs["xc_graph"] + xs_detached
            outputs["logits_ba"] = self.classifier(combined)
        
        outputs["logits_gnn"] = self.classifier(outputs["xc_graph"])
        outputs["logits"] = self.classifier(xc)
        return outputs


## 4. Graph Builder & Memory Trainer

In [ ]:
def build_knn_graph(features, k=5, drop_edge_p=0.0, training=False, temperature=0.1, pseudo_labels=None, heterophily_scorer=None):
    batch_size = features.size(0)
    feat_norm = F.normalize(features, p=2, dim=1)
    sim_matrix = torch.mm(feat_norm, feat_norm.t())
    sim_matrix.fill_diagonal_(-9999.0)
    
    topk_sim, topk_indices = torch.topk(sim_matrix, k=k, dim=1)
    
    if heterophily_scorer is not None and pseudo_labels is not None:
        row_indices = torch.arange(batch_size, device=features.device).view(-1, 1).expand(-1, k)
        i_idx = row_indices.flatten()
        j_idx = topk_indices.flatten()
        h_score = heterophily_scorer(features[i_idx], features[j_idx], pseudo_labels[i_idx], pseudo_labels[j_idx])
        h_score = h_score.view(batch_size, k)
        # The Edge Penalty Logic
        topk_sim = topk_sim + temperature * torch.log(h_score + 1e-8)
    
    weights = F.softmax(topk_sim / temperature, dim=1)
    adj = torch.zeros(batch_size, batch_size, device=features.device)
    adj.scatter_(1, topk_indices, weights)
    
    if training and drop_edge_p > 0.0:
        mask = torch.rand_like(adj) > drop_edge_p
        adj = adj * mask
        row_sum = adj.sum(dim=1, keepdim=True).clamp(min=1e-8)
        adj = adj / row_sum
    
    adj = adj + torch.eye(batch_size, device=features.device)
    row_sum2 = adj.sum(dim=1, keepdim=True).clamp(min=1e-8)
    return adj / row_sum2

class MemoryBank:
    def __init__(self, size, dim, device):
        self.size = size
        self.dim = dim
        self.device = device
        self.bank = torch.zeros(size, dim, device=device)
        self.ptr = 0
        self.is_full = False
        
    def update(self, features):
        batch_size = features.size(0)
        features = features.detach().clone()
        if batch_size >= self.size:
            self.bank = features[-self.size:]
            self.ptr = 0
            self.is_full = True
            return
            
        end_idx = self.ptr + batch_size
        if end_idx <= self.size:
            self.bank[self.ptr:end_idx] = features
            self.ptr = end_idx
        else:
            overflow = end_idx - self.size
            self.bank[self.ptr:] = features[:batch_size-overflow]
            self.bank[:overflow] = features[batch_size-overflow:]
            self.ptr = overflow
            
        if self.ptr >= self.size:
            self.ptr = 0
            self.is_full = True
            
    def sample(self, M=4):
        if not self.is_full and self.ptr == 0:
            return torch.zeros(M, self.dim, device=self.device)
        max_idx = self.size if self.is_full else self.ptr
        indices = torch.randint(0, max_idx, (M,), device=self.device)
        return self.bank[indices]

class StandaloneTrainer:
    def __init__(self, model, optimizer, device, max_epochs, k_neighbors=5, memory_size=256, m_samples=4):
        self.model = model
        self.optimizer = optimizer
        self.device = device
        self.max_epochs = max_epochs
        self.k_neighbors = k_neighbors
        self.m_samples = m_samples
        self.criterion_cls = nn.CrossEntropyLoss()
        self.criterion_domain = nn.CrossEntropyLoss()
        
        spurious_dim = model.causal_disentangle.spurious_enc[-1].out_features
        self.memory_bank = MemoryBank(size=memory_size, dim=spurious_dim, device=device)

    def train_epoch(self, dataloader, epoch):
        self.model.train()
        total_loss = 0
        all_preds, all_targets = [], []
        all_probs = []
        
        alpha_gnn = min(0.3, 0.3 * (epoch / 15.0))
        grl_lambda = get_grl_lambda(epoch, self.max_epochs)
        
        for batch in dataloader:
            if len(batch) == 4:
                img_feat, txt_feat, labels, domains = [b.to(self.device) for b in batch]
            else:
                img_feat, txt_feat, labels = [b.to(self.device) for b in batch]
                domains = torch.zeros_like(labels)
            
            self.optimizer.zero_grad()
            out_draft = self.model(img_feat, txt_feat)
            
            # Update Spurious Memory Bank
            self.memory_bank.update(out_draft["xs"])
            
            # GT Pseudo Labels for Heterophily Scorer during Training
            num_classes = self.model.classifier[-1].out_features
            gt_labels = F.one_hot(labels, num_classes=num_classes).float()
            adj = build_knn_graph(out_draft["xc"], self.k_neighbors, drop_edge_p=0.3, training=True, 
                                  pseudo_labels=gt_labels, heterophily_scorer=self.model.heterophily_scorer)
            
            out = self.model(img_feat, txt_feat, adj=adj, backdoor_xs=None)
            
            loss_task_p1 = self.criterion_cls(out["logits"], labels)
            loss_task_gnn = self.criterion_cls(out["logits_ba"], labels)
            
            xc_reversed = GradientReversalFunction.apply(out["xc_graph"], grl_lambda)
            domain_logits_adv = self.model.domain_classifier(xc_reversed)
            loss_disc = self.criterion_domain(domain_logits_adv, domains)
            
            # Simplified Total Loss for Standalone
            loss = loss_task_p1 + alpha_gnn * loss_task_gnn + 0.01 * loss_disc
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 2.0)
            self.optimizer.step()
            
            total_loss += loss.item()
            all_preds.extend(torch.argmax(out["logits_ba"], dim=1).cpu().numpy())
            all_targets.extend(labels.cpu().numpy())
            
        epoch_f1 = f1_score(all_targets, all_preds, average='weighted')
        return total_loss / len(dataloader), epoch_f1

    @torch.no_grad()
    def evaluate(self, dataloader):
        self.model.eval()
        total_loss = 0
        all_preds, all_targets = [], []
        all_probs = []
        
        for batch in dataloader:
            if len(batch) == 4:
                img_feat, txt_feat, labels, _ = [b.to(self.device) for b in batch]
            else:
                img_feat, txt_feat, labels = [b.to(self.device) for b in batch]
                
            out_draft = self.model(img_feat, txt_feat)
            
            # Pseudo Logits for Edge Heterophily Penalty
            pseudo_probs = torch.softmax(out_draft["logits"], dim=-1)
            adj = build_knn_graph(out_draft["xc"], self.k_neighbors, drop_edge_p=0.0, training=False, 
                                  pseudo_labels=pseudo_probs, heterophily_scorer=self.model.heterophily_scorer)
            
            backdoor_xs = None
            if self.memory_bank.is_full or self.memory_bank.ptr > 0:
                bank_samples = self.memory_bank.sample(M=self.m_samples)
                backdoor_xs = bank_samples.unsqueeze(0).expand(img_feat.size(0), self.m_samples, -1)
            else:
                backdoor_xs = out_draft["xs"].unsqueeze(1)
                
            out = self.model(img_feat, txt_feat, adj=adj, backdoor_xs=backdoor_xs)
            loss = self.criterion_cls(out["logits_ba"], labels)
            probs = torch.softmax(out["logits_ba"], dim=1)
            all_probs.extend(probs.cpu().numpy())
            preds = torch.argmax(out["logits_ba"], dim=1)
            
            total_loss += loss.item()
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())
            
        bAcc = accuracy_score(all_targets, all_preds)
        f1 = f1_score(all_targets, all_preds, average='weighted')
                all_probs_np = np.array(all_probs)
        try:
            if all_probs_np.shape[1] == 2:
                auc = roc_auc_score(all_targets, all_probs_np[:, 1])
            else:
                auc = roc_auc_score(all_targets, all_probs_np, multi_class='ovr', average='weighted')
        except Exception:
            auc = 0.0
            
        return total_loss / len(dataloader), f1, bAcc, all_preds, all_targets, auc


## 5. Fetch Embeddings & Dataset Injection 
Thay thế đoạn `load_numpy()` này bằng file trích xuất CLIP thực tế trên Drive của bạn, hoặc tự sinh npy ngẫu nhiên để test flow.

In [ ]:
# TEST MOCK DATASET (Đảm bảo GNN flow không gãy Tensor)
BATCH_SIZE = 256
N_TRAIN = 1000
N_TEST = 200
NUM_CLASSES = 8
NUM_DOMAINS = 7

# Mock CLIP Features
train_img = np.random.randn(N_TRAIN, 1024)
train_txt = np.random.randn(N_TRAIN, 768)
train_labels = np.random.randint(0, NUM_CLASSES, size=(N_TRAIN,))
train_domain_labels = np.random.randint(0, NUM_DOMAINS, size=(N_TRAIN,))

test_img = np.random.randn(N_TEST, 1024)
test_txt = np.random.randn(N_TEST, 768)
test_labels = np.random.randint(0, NUM_CLASSES, size=(N_TEST,))
test_domain_labels = np.random.randint(0, NUM_DOMAINS, size=(N_TEST,))

train_dataset = TensorDataset(
    torch.tensor(train_img, dtype=torch.float32),
    torch.tensor(train_txt, dtype=torch.float32),
    torch.tensor(train_labels, dtype=torch.long),
    torch.tensor(train_domain_labels, dtype=torch.long)
)
test_dataset = TensorDataset(
    torch.tensor(test_img, dtype=torch.float32),
    torch.tensor(test_txt, dtype=torch.float32),
    torch.tensor(test_labels, dtype=torch.long),
    torch.tensor(test_domain_labels, dtype=torch.long)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
print("Mock Data Loaded!")

## 6. Training Execution

In [ ]:
model = CausalCrisisV2Model(
    img_dim=train_img.shape[1],
    txt_dim=train_txt.shape[1],
    hidden_dim=256,
    causal_dim=256,
    spurious_dim=256,
    num_domains=NUM_DOMAINS,
    num_classes=NUM_CLASSES,
    dropout=0.5
).to(device)

optimizer = torch.optim.AdamW([
    {'params': [p for n, p in model.named_parameters() if 'gnn' not in n and 'heterophily' not in n], 'weight_decay': 1e-4},
    {'params': [p for n, p in model.named_parameters() if 'gnn' in n or 'heterophily' in n], 'weight_decay': 1e-3} 
], lr=5e-4)

EPOCHS = 10
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

trainer = StandaloneTrainer(model, optimizer, device, max_epochs=EPOCHS, k_neighbors=5, memory_size=256, m_samples=4)

print("Starting Training Causal GNN + Heterophily Scorer...")
for epoch in range(1, EPOCHS + 1):
    train_loss, train_f1 = trainer.train_epoch(train_loader, epoch)
    test_loss, test_f1, test_acc, _, _, test_auc = trainer.evaluate(test_loader)
    scheduler.step()
    print(f"Epoch {epoch:2d}/{EPOCHS} | Train F1: {train_f1:.4f} | Test F1: {test_f1:.4f} | bAcc: {test_acc:.4f} | AUC: {test_auc:.4f}")


## 7. Metrics & Visualizations

In [ ]:
test_loss, test_f1, test_acc, all_preds, all_targets = trainer.evaluate(test_loader)
cm = confusion_matrix(all_targets, all_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix (CausalCrisis v2 + Heterophily GNN)")
plt.ylabel("Ground Truth")
plt.xlabel("Predictions")
plt.show()

print("\nClassification Report:")
print(classification_report(all_targets, all_preds))